### Importing Python Packages and Utility Functions

In [1]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

### Data Loading and Structure

The dataset consists of three key files:


- **features.parquet**: A DataFrame containing **22 stock features** for each trading day from **2005 to 2025**, stored in a **columnar fashion**. For example, `features["macd"]` will return the `macd` of shape **(5068, 2167)**. All **22** features are guaranteed to have the same shape.

- **universe.parquet**: A DataFrame of the same shape as `features["macd"]`, containing `0` and `1` values, where `1` indicates that the stock is tradable on that day.

- **returns.parquet**: A DataFrame of shape **(3775, 2167)** containing **daily stock returns** from **2005 to 2019**. You will **never receive** the returns for the testing period i.e. from **2020 to 2025**.

### Data Organization:
- **Columns** represent stock identifiers, ranging from **1 to 2167** in increasing order.
  
- **Rows** represent trading days when the market was open.  


In [4]:
# Change this to your data directory
data_dir = "/Users/adityaanand/dev/data/"

features = pd.read_parquet(os.path.join(data_dir, "features.parquet"))

universe = pd.read_parquet(os.path.join(data_dir, "universe.parquet"))
 
returns = pd.read_parquet(os.path.join(data_dir, "returns.parquet"))


### Benchmark Strategy: Vectorized Portfolio Generation

In this step we will write a vectorized code to generate our strategy portfolio for all trading days at once without a `for` loop. For doing this we will use direct operations on feature dataframes. But, here's one caution. It is very easy to look into the future feature data when you're using dataframes to construct your portfolio at once.

Below is a vectorized implementation of the benchmark strategy [which you see on Kaggle]. Make sure to note the `alpha.shift(1)` operation in the code! This is to make sure you use only feature values upto the last trading day to construct today's portfolio weights.

You can change the code below to implement your own strategy. Since this vectorized implementation is faster, you can try various versions of your strategy and see their performances.

#### Inputs:
- `entire_features :pd.DataFrame`: Historical feature data (MultiIndex columns: Feature Names, Stock Identifiers).
- `universe: pd.DataFrame`: Binary DataFrame indicating tradable stocks per day.
- `start_date`, `end_date` :`str`: Backtest period in `'YYYY-MM-DD'` format.

#### Output:
- `portfolio(pd.DataFrame)`: Normalized portfolio weights for each stock per day in the specified date range.


In [ ]:
import strategy
importlib.reload(strategy)
from strategy import generate_portfolio_vectorized, get_weights, SIGNAL_CONFIGS, FEAT_EWMA, PORTFOLIO_EWMA, VOL_GROUPS, CONCENTRATION_PCT

print(f"Strategy: {len(SIGNAL_CONFIGS)} features, feat_EWMA={FEAT_EWMA}, port_EWMA={PORTFOLIO_EWMA}")
print(f"  Vol groups={VOL_GROUPS}, concentration={CONCENTRATION_PCT:.0%}")

### Generate your portfolio using the `generate_portfolio_vectorized` function you wrote above

In [6]:
benchmark_portfolio_vectorized = generate_portfolio_vectorized(
    features,
    universe,
    "2005-01-03",
    "2025-02-07"
)

### Backtest your portfolio generated using vectorized code

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date. 

In [7]:
sr_vectorized, pnl_vectorized = backtest_portfolio(benchmark_portfolio_vectorized.loc[:"2019"], returns.loc[:"2019"], universe.loc[:"2019"], True, True)

Gross Sharpe Ratio:  1.32
Net Sharpe Ratio:  1.11
Turnover %:  31.872


### Benchmark Strategy: Iterative Portfolio Generation

Although the Vectorized Function generated the portfolio very quickly, it is very easy to look into the future data if you are not careful. For instance, remove the `shift(1)` operation and see the performance of the portfolio 😊. Hence, if your vectorized code has a forward bias [lookahead bias], you may see very high [or very low] sharpe ratios which may never be realised in real trading.

To avoid making these mistakes, we simulate our portfolio in a daily iterative fashion, where we call the `get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]` function with **only** the past features data and the current day's trading universe. 

### Function Inputs:
- `features(pd.DataFrame)`:  
  - Contains various stock features indexed by date and stock identifiers.  
  - The features are structured in a **MultiIndex format**, where level 0 represents **feature names** (e.g., "macd", "volatility_60"), and level 1 represents **stock identifiers** (e.g., "1", "2", ..., "2167").  

- `today_universe(pd.Series)`:  
  - A series indicating which stocks can be traded on the current day.  
  - Contains **binary values (0 or 1)**, where **1** means a stock is **tradable**, and **0** means it is not.

You have to change this code, and write your own strategy code inside this function. Make sure it follows the same semantics as explained above.


In [ ]:
# get_weights is imported from strategy.py
# See strategy.py for the full implementation including:
# - 22-feature IC_IR-weighted signal combination
# - Portfolio EWMA smoothing
# - Volatility-group neutralization (10 groups)
# - Signal concentration (top/bottom 20%)
print("get_weights imported from strategy.py")

### Generate your portfolio using the `generate_portfolio` and `get_weights` function you wrote above

Since this function is iteratively called on every trading day, it takes a lot of time [about 40 mintues] to generate the entire portfolio dataframe from `2005-01-03` to `2025-02-07`. Hence, to show an example we call it only for a one year period from `2010-01-01` to `2010-12-31`.

In [9]:
benchmark_portfolio = generate_portfolio(
    get_weights,
    features,
    universe,
    "2010-01-01",
    "2010-12-31",
)

100%|██████████| 252/252 [01:10<00:00,  3.58it/s]


### Backtest your portfolio

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date. 

In [10]:
sr, pnl = backtest_portfolio(benchmark_portfolio.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True)

Gross Sharpe Ratio:  1.177
Net Sharpe Ratio:  0.866
Turnover %:  31.573


You can also check the performance of your vectorized portfolio in this period to see if they match!

In [11]:
sr, pnl = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True)

Gross Sharpe Ratio:  1.177
Net Sharpe Ratio:  0.866
Turnover %:  31.573


### Comparing Iterative and Vectorized Portfolio Implementations
This function evaluates **iterative** and **vectorized** portfolio generation methods by comparing their PnL correlation. If **correlation ≥ 0.98**, both implementations are considered equivalent.
#### Steps:
1. Selects a random start date.
2. Generates portfolios using both methods.
3. Backtests portfolios and computes PnLs.
4. Validates PnL correlation.
#### Criteria:
- **Pass:** Correlation **≥ 0.98** (implementations match).
- **Fail:** Correlation **< 0.98** (error raised).
#### Inputs:
- `contestant_get_weights`: Function for portfolio weights.
- `contestant_vectorized_portfolio`: A Pandas DataFrame containing portfolio weights generated using Vectorized Implementation
- `entire_features`: Feature data (MultiIndex columns).
- `universe`: Tradable stocks (binary).
- `returns`: Daily stock returns.
#### Output:
- Prints PnL correlation or raises an error if mismatched.

In [12]:
match_implementations(get_weights, benchmark_portfolio_vectorized, features, universe, returns)

Starting to generate Iterative Portfolio


100%|██████████| 41/41 [00:19<00:00,  2.11it/s]


Iterative Portfolio Generated
Correlation of 1.0 between Iterative and Vectorized Implementations. Both implementations match!


### Final Notes
- We recommend using vectorized code to test out your strategies. This will be easier and faster to run but make sure to `shift(1)` feature dataframes in order to avoid lookahead or forward bias.
- At the end when you have decided your final strategy that you want to submit for the competition, we advise you to write code for `get_weights` which will help iteratively generate your portfolio. 
- Finally, before submitting make sure to run `match_implementations` to make sure that both versions of your code produce the same portfolio
- If these two portfolios match, you can submit the one which was produced by `generate_portfolio_vectorized` without waiting for the iterative portfolio. You don't need to run the `generate_portfolio` function for 20 years!
- We will check that the submission you made on Kaggle matches with the portfolio generated by your code. If these two don't match, you will be eliminated from the competition. 

In [68]:
# Submit this csv file on kaggle
benchmark_portfolio_vectorized.to_csv("submission.csv")